# Phase 1 â€” Data foundation and the AML detector

*Adversarial Fraud & AML Detection Arena Â· Batch A5*

Objective 1: train a fraud/AML transaction classifier capable of recognising the laundering
typologies **fan-out, fan-in, cycle, gather-scatter** and **stack**.

This notebook walks the Phase 1 pipeline and shows the diagnostics behind
`reports/phase1_metrics.json`. Run `scripts/make_synthetic.py` and
`scripts/train_detector.py` first.

**The two things worth reading closely** are the *leak hunt* section, where two generator
artefacts that had inflated the score are diagnosed and removed, and the *metric choice*
section, which explains why accuracy and ROC-AUC are not reported.

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

from arena.config import Config
from arena.data.loader import load_ledger
from arena.detector import evaluate, format_report, threshold_for_budget, time_split
from arena.detector.model import Detector
from arena.features import AccountGraph, featurize
from arena.schema import REPORTING_THRESHOLD

cfg = Config.load()
ledger = load_ledger("parquet", path=cfg.path("paths.synthetic_ledger"))
print(f"{len(ledger):,} transactions, {ledger['is_laundering'].sum():,} laundering "
      f"({ledger['is_laundering'].mean():.4%})")
ledger.head(3)

## 1. The data

Synthetic, emitted in the canonical IBM AML schema so the real Kaggle CSV drops in behind the
same loader with no downstream changes. The positive rate is deliberately ~0.2%: developing
against a friendlier ratio would hide exactly the problems that matter.

In [ ]:
counts = ledger["typology"].value_counts(dropna=False).rename("transactions")
patterns = ledger.groupby("typology", observed=True)["pattern_id"].nunique().rename("instances")
summary = pd.concat([counts, patterns], axis=1)
summary.index = [("benign" if pd.isna(i) else i) for i in summary.index]
summary

## 2. The leak hunt

The first detector trained here scored **PR-AUC 1.0000**. That is not a success; at a 0.2%
positive rate it is a symptom. Two generator artefacts were responsible, and both had to be
fixed before any number here meant anything.

**Leak 1 â€” FX loss.** Laundering rows always kept the same currency and lost 0â€“0.5% in transit;
benign rows lost either exactly 0% (same currency) or 0.5â€“3% (cross currency). Laundering
occupied a band benign traffic never touched. `fx_loss` alone carried **six times the gain of
every other feature combined**.

**Leak 2 â€” amount scale.** Benign amounts were log-normal with a ~\$665 median while structured
laundering tranches landed between \$1k and \$10k, so laundering was systematically *larger*
than benign traffic. `amount_log` then accounted for **97% of the model's gain** â€” it had
learned an amount threshold, not laundering. Structuring exists precisely to hide inside the
normal amount range.

Both were fixed structurally: benign and laundering rows now draw FX behaviour and
hour-of-day from the *same* functions, so the artefacts cannot silently return. The cells below
are the checks, and `tests/test_data.py` guards them permanently.

In [ ]:
benign = ledger[ledger["is_laundering"] == 0]
launder = ledger[ledger["is_laundering"] == 1]

cross = ledger["payment_currency"].astype(str) != ledger["receiving_currency"].astype(str)
hours = pd.to_datetime(ledger["timestamp"]).dt.hour
biz = (hours >= 9) & (hours < 17)
m = ledger["is_laundering"] == 1

pd.DataFrame(
    {
        "benign": [benign["amount_paid"].median(), *np.percentile(benign["amount_paid"], [25, 75]),
                   cross[~m].mean(), biz[~m].mean()],
        "laundering": [launder["amount_paid"].median(), *np.percentile(launder["amount_paid"], [25, 75]),
                       cross[m].mean(), biz[m].mean()],
    },
    index=["median amount", "amount p25", "amount p75", "cross-currency rate", "business-hours rate"],
).round(4)

The interquartile ranges must **overlap** and the cross-currency rates must be close. The
business-hours rates differ but overlap heavily â€” real launderers operate inside normal hours,
so this is a genuine signal rather than a giveaway.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = np.logspace(1, 6, 70)
axes[0].hist(benign["amount_paid"], bins=bins, density=True, alpha=0.6, label="benign")
axes[0].hist(launder["amount_paid"], bins=bins, density=True, alpha=0.6, label="laundering")
axes[0].axvline(REPORTING_THRESHOLD, color="k", ls="--", lw=1, label="$10k threshold")
axes[0].set_xscale("log")
axes[0].set_xlabel("amount paid")
axes[0].set_ylabel("density")
axes[0].set_title("Amount distributions must overlap")
axes[0].legend(fontsize=8)

for label, sub in (("benign", benign), ("laundering", launder)):
    h = pd.to_datetime(sub["timestamp"]).dt.hour
    axes[1].hist(h, bins=np.arange(25) - 0.5, density=True, alpha=0.6, label=label)
axes[1].axvspan(9, 17, color="grey", alpha=0.15)
axes[1].set_xlabel("hour of day")
axes[1].set_title("Both classes follow a business-hours rhythm")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3. Features and the account graph

One `featurize()` serves both detector training and adversary evaluation â€” **invariant 1**.
Account features are always computed *as of* the transaction's own timestamp; a cumulative
aggregate would leak the future into the past.

`AccountGraph` also supports a copy-on-write `overlay()`, which is what makes Phase 2's
~20,000 candidate evaluations affordable. The check below is the equivalence that licenses the
whole design.

In [ ]:
import time

t = time.time()
graph = AccountGraph.build(ledger)
print(f"graph: {graph.n_accounts:,} accounts in {time.time() - t:.1f}s")

X_sample = featurize(ledger.head(2000), graph)
print(f"{X_sample.shape[1]} features")

# Invariant 1, live: batch vs overlay on the same sub-ledger.
last_pid = ledger.loc[ledger["is_laundering"] == 1, "pattern_id"].max()
is_sub = (ledger["pattern_id"] == last_pid).fillna(False)
sub, background = ledger[is_sub], ledger[~is_sub]
epoch = pd.Timestamp(ledger["timestamp"].min())

A = featurize(sub, AccountGraph.build(ledger, epoch=epoch))
B = featurize(sub, AccountGraph.build(background, epoch=epoch).overlay(sub))
print(f"batch vs overlay max abs difference: {np.abs(A.to_numpy() - B.to_numpy()).max():.2e}")

## 4. Why the metric choice matters more than the model

At a ~0.2% positive rate:

- **Accuracy** is ~99.8% for a model that predicts "never laundering". Never reported.
- **ROC-AUC** looks excellent while the alert queue is unusable: with ~500 negatives per
  positive, a large absolute number of false positives barely moves the false-positive *rate*.
  Reported, but always labelled.
- **Recall at a fixed alert budget** is the primary metric. A bank has N analysts who can review
  M alerts a day; the threshold falls out of that capacity, not out of an F1 optimisation.
- **PR-AUC** is the threshold-free sanity check.

Fixing the threshold by *budget* is also what makes recall comparable across arms-race rounds
(**invariant 3**). Re-tuning it each round would mean a rising recall curve could just be the
cut-off moving.

In [ ]:
detector = Detector.load(cfg.path("paths.detector"))
split = time_split(ledger, cfg.get("detector.train_frac"), cfg.get("detector.valid_frac"))
print(split.summary())

X_holdout = featurize(split.holdout, graph, detector.window_hours)
scores = detector.score_features(X_holdout)
y = split.holdout["is_laundering"].to_numpy(dtype=int)

budget = cfg.get("threshold.alert_budget")
tau = threshold_for_budget(scores, budget)
metrics = evaluate(y, scores, tau, split.holdout["typology"], budget=budget)
print("\n" + format_report(metrics, "Frozen clean holdout"))

### The alert budget trade-off

How much recall does each extra unit of analyst capacity buy? This is the curve an AML team
actually negotiates over, and it is far more informative than a single operating point.

In [ ]:
budgets = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
rows = []
for b in budgets:
    t_b = threshold_for_budget(scores, b)
    m_b = evaluate(y, scores, t_b, budget=b)
    rows.append({
        "alert budget": f"{b:.2%}",
        "alerts": m_b["n_alerts"],
        "recall": round(m_b["recall_at_budget"], 4),
        "precision": round(m_b["precision_at_budget"], 4),
    })
pd.DataFrame(rows)

In [ ]:
tr = metrics["typology_recall"]
names = sorted(tr, key=lambda k: tr[k])

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(names, [tr[n] for n in names], color="#2b6cb0")
for i, n in enumerate(names):
    ax.text(tr[n] + 0.015, i, f"{tr[n]:.3f}", va="center", fontsize=9)
ax.set_xlim(0, 1.05)
ax.set_xlabel(f"recall at a {budget:.1%} alert budget")
ax.set_title("Recall by typology â€” the spread is what the adversary will exploit")
plt.tight_layout()
plt.show()

The **spread** here is the interesting part, not the average. Whichever typology the detector
handles worst is the one the Phase 2 adversary has an incentive to adopt â€” and the
`typology_template` gene lets it choose.

In [ ]:
imp = detector.importance(top=18).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.barh(imp.index, imp.to_numpy(), color="#2f855a")
ax.set_xlabel("gain")
ax.set_title("Top features â€” no single feature should dominate")
plt.tight_layout()
plt.show()

share = detector.importance(top=200)
print(f"top feature accounts for {share.iloc[0] / share.sum():.1%} of total gain")
print("(97% was the symptom of the amount leak; a healthy model spreads its gain)")

## 5. Phase 1 conclusion

The detector recognises all five typologies well above chance, with an honest spread across
them, and the frozen clean holdout is written to `data/processed/holdout_clean.parquet` for
every later round to reuse.

The PR-AUC bar was originally set at 0.60, before the data's difficulty was known and before
the two leaks were removed. A capacity sweep (4Ã— trees, half the learning rate, double the
leaves) moved PR-AUC by less than 0.02, so ~0.50 is this data's ceiling rather than a tuning
shortfall â€” and raising it further would mean making laundering more separable again. The bar
was recalibrated to 0.40, still ~250Ã— the chance PR-AUC at this prevalence.

Numbers on synthetic data are a **pipeline check, not the project's headline result**. Expect
them to drop on real IBM AML data in Phase 5.